# E5.2 — Sensitivity Analysis (OAT Parameter Variation)

Analyses how robust the Pareto front shape and portfolio compositions (host assignment, development timing, well count) are to changes in input assumptions.

**Method:** One-At-a-Time (OAT) sensitivity analysis. Ten parameter groups are varied while holding all others at default values. Each parameter group is tested with 10 independent seeds. The final configuration from E1.1–E4.1 (Gray encoding, FCFS capping, LSMC-based ROV, DSBPSO) is used throughout.

| Group | Variants | Parameter | Baseline |
|-------|----------|-----------|----------|
| Discount rate | 4%, 6%, 10%, 12% | `valuation.discount_rate` | 8% |
| CVaR alpha | 1%, 10%, 20% | `valuation.cvar_alpha` | 5% |
| Volatility | 0.5×, 0.75×, 1.25×, 1.5× | Price sigma scale | 1.0× |
| Host capacity | 0.5×, 0.75×, 1.5×, 2.0× | Capacity scale | 1.0× |
| Field size | 0.5×, 0.75×, 1.25×, 1.5× | Oil-in-place scale | 1.0× |
| Oil price | $52, $61, $82, $110/bbl | Oil E₀ shift | ~$71/bbl |
| CAPEX (all) | 0.5×–4.0× | All CAPEX components | 1.0× |
| Well cost | 0.5×, 1.5×, 2.0×, 4.0× | Drilling cost only | 1.0× |
| Subsea | 0.5×, 1.5×, 2.0×, 4.0× | Subsea infrastructure only | 1.0× |
| Pipeline | 0.5×, 1.5×, 2.0×, 4.0× | Pipeline cost only | 1.0× |

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

from src.experiments.runner import load_experiment_results, load_run_result
from src.experiments.metrics import hypervolume_2d
from src.optimizer.encoding import decode_particle
from src.data_generation.generator import generate_data
from src.config import load_config

EXPERIMENT_ID = "E5.2"

In [ ]:
# Parameter group metadata: config names (with baseline inserted at correct position),
# numeric values for x-axis, and display info.
PARAM_GROUPS = {
    "Discount rate": {
        "configs": ["discount_4pct", "discount_6pct", "baseline", "discount_10pct", "discount_12pct"],
        "values": [4, 6, 8, 10, 12],
        "baseline_idx": 2,
        "xlabel": "Discount rate (%)",
    },
    "CVaR alpha": {
        "configs": ["cvar_1pct", "baseline", "cvar_10pct", "cvar_20pct"],
        "values": [1, 5, 10, 20],
        "baseline_idx": 1,
        "xlabel": r"CVaR $\alpha$ (%)",
    },
    "Volatility": {
        "configs": ["volatility_0p5x", "volatility_0p75x", "baseline", "volatility_1p25x", "volatility_1p5x"],
        "values": [0.5, 0.75, 1.0, 1.25, 1.5],
        "baseline_idx": 2,
        "xlabel": "Volatility scale",
    },
    "Host capacity": {
        "configs": ["capacity_0p5x", "capacity_0p75x", "baseline", "capacity_1p5x", "capacity_2p0x"],
        "values": [0.5, 0.75, 1.0, 1.5, 2.0],
        "baseline_idx": 2,
        "xlabel": "Host capacity scale",
    },
    "Field size": {
        "configs": ["fieldsize_0p5x", "fieldsize_0p75x", "baseline", "fieldsize_1p25x", "fieldsize_1p5x"],
        "values": [0.5, 0.75, 1.0, 1.25, 1.5],
        "baseline_idx": 2,
        "xlabel": "Field size scale",
    },
    "Oil price": {
        "configs": ["oilprice_52usd", "oilprice_61usd", "baseline", "oilprice_82usd", "oilprice_110usd"],
        "values": [52, 61, 71, 82, 110],
        "baseline_idx": 2,
        "xlabel": "Oil price (USD/bbl)",
    },
    "CAPEX": {
        "configs": ["capex_0p5x", "capex_0p75x", "baseline", "capex_1p25x", "capex_1p5x",
                     "capex_2p0x", "capex_3p0x", "capex_4p0x"],
        "values": [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0],
        "baseline_idx": 2,
        "xlabel": "CAPEX scale (all components)",
    },
    "Well cost": {
        "configs": ["wellcost_0p5x", "baseline", "wellcost_1p5x", "wellcost_2p0x", "wellcost_4p0x"],
        "values": [0.5, 1.0, 1.5, 2.0, 4.0],
        "baseline_idx": 1,
        "xlabel": "Well cost scale",
    },
    "Subsea": {
        "configs": ["subsea_0p5x", "baseline", "subsea_1p5x", "subsea_2p0x", "subsea_4p0x"],
        "values": [0.5, 1.0, 1.5, 2.0, 4.0],
        "baseline_idx": 1,
        "xlabel": "Subsea cost scale",
    },
    "Pipeline": {
        "configs": ["pipeline_0p5x", "baseline", "pipeline_1p5x", "pipeline_2p0x", "pipeline_4p0x"],
        "values": [0.5, 1.0, 1.5, 2.0, 4.0],
        "baseline_idx": 1,
        "xlabel": "Pipeline cost scale",
    },
}

GROUP_COLORS = {
    "Discount rate": "steelblue",
    "CVaR alpha": "orchid",
    "Volatility": "coral",
    "Host capacity": "seagreen",
    "Field size": "goldenrod",
    "Oil price": "sienna",
    "CAPEX": "slategray",
    "Well cost": "darkred",
    "Subsea": "teal",
    "Pipeline": "mediumpurple",
}


def group_slug(group_name: str) -> str:
    """Convert group name to filename-safe slug."""
    return group_name.lower().replace(" ", "_")


def pct_change(value: float, baseline: float) -> float:
    """Percentage change relative to baseline."""
    return (value - baseline) / abs(baseline) * 100

## 2. Load Results

In [ ]:
runs = load_experiment_results(EXPERIMENT_ID)
configs = defaultdict(list)
for run in runs:
    s = run["summary"]
    if s["status"] != "completed":
        print(f"  SKIPPING failed run: {s['run_id']}")
        continue
    configs[s["config_name"]].append(run)

print(f"Loaded {sum(len(v) for v in configs.values())} completed runs across {len(configs)} configs\n")
for name in sorted(configs.keys()):
    times = [r["summary"]["elapsed_seconds"] for r in configs[name]]
    print(f"  {name:25s}: {len(configs[name]):2d} seeds, avg time {np.mean(times):7.1f}s")

In [ ]:
# Extract best E[ROV] and best CVaR from JSON summaries (no pickle loading needed)
summary_stats = {}
for name, run_list in configs.items():
    rovs = np.array([r["summary"]["best_rov"] for r in run_list])
    cvars = np.array([r["summary"]["best_cvar"] for r in run_list])
    summary_stats[name] = {"best_rov": rovs, "best_cvar": cvars}

bl_rov = summary_stats["baseline"]["best_rov"].mean()
bl_cvar = summary_stats["baseline"]["best_cvar"].mean()
print(f"Baseline: E[ROV] = {bl_rov:,.0f} MNOK, CVaR = {bl_cvar:,.0f} MNOK")

## 3. Summary Table

In [ ]:
# Summary table: mean objectives and % change from baseline for each config
print(f"{'Config':<25s}  {'Group':<15s}  {'E[ROV] mean':>12s}  {'std':>8s}  {'ΔROV%':>7s}"
      f"  {'CVaR mean':>11s}  {'std':>8s}  {'ΔCVaR%':>7s}")
print("-" * 115)

for group_name, group_info in PARAM_GROUPS.items():
    for cfg in group_info["configs"]:
        if cfg == "baseline":
            continue
        stats = summary_stats[cfg]
        m_rov = stats["best_rov"].mean()
        s_rov = stats["best_rov"].std()
        m_cvar = stats["best_cvar"].mean()
        s_cvar = stats["best_cvar"].std()
        d_rov = pct_change(m_rov, bl_rov)
        d_cvar = pct_change(m_cvar, bl_cvar)
        # Mark CVaR alpha changes as metric redefinition
        cvar_note = " *" if group_name == "CVaR alpha" else ""
        print(f"{cfg:<25s}  {group_name:<15s}  {m_rov:>12,.0f}  {s_rov:>8,.0f}  {d_rov:>+6.1f}%"
              f"  {m_cvar:>11,.0f}  {s_cvar:>8,.0f}  {d_cvar:>+6.1f}%{cvar_note}")
    print()

print("* CVaR alpha changes redefine the risk metric (different tail quantile), not the portfolio.")

## 4. Tornado Diagram — Parameter Sensitivity Overview

For each parameter group, shows the range of % change in E[ROV] and CVaR relative to baseline, spanning from the group's lowest to highest variant. CVaR alpha is excluded because it redefines the metric rather than changing the portfolio.

In [ ]:
# Compute tornado data: min/max % change per group (excluding CVaR alpha)
tornado_data = {}
for group_name, group_info in PARAM_GROUPS.items():
    if group_name == "CVaR alpha":
        continue

    pct_rov = []
    pct_cvar = []
    for cfg in group_info["configs"]:
        if cfg == "baseline":
            continue
        m_rov = summary_stats[cfg]["best_rov"].mean()
        m_cvar = summary_stats[cfg]["best_cvar"].mean()
        pct_rov.append(pct_change(m_rov, bl_rov))
        pct_cvar.append(pct_change(m_cvar, bl_cvar))

    tornado_data[group_name] = {
        "rov_min": min(pct_rov),
        "rov_max": max(pct_rov),
        "cvar_min": min(pct_cvar),
        "cvar_max": max(pct_cvar),
        "total_swing": (max(pct_rov) - min(pct_rov)) + (max(pct_cvar) - min(pct_cvar)),
    }

In [ ]:
# Pastel palette (lighter cousins of the standard steelblue / coral / seagreen / orchid / goldenrod scheme)
TORNADO_COLORS = {
    "Discount rate": "lightsteelblue",
    "Volatility":    "lightsalmon",
    "Host capacity": "darkseagreen",
    "Field size":    "khaki",
    "Oil price":     "burlywood",
    "CAPEX":         "lightgray",
    "Well cost":     "rosybrown",
    "Subsea":        "paleturquoise",
    "Pipeline":      "thistle",
}

# Rank groups by total swing for consistent ordering across both panels
ranked = sorted(tornado_data.items(), key=lambda x: x[1]["total_swing"])
group_names = [name for name, _ in ranked]
n_groups = len(group_names)


def _draw_tornado(ax, metric, label):
    y_pos = np.arange(n_groups)
    for i, name in enumerate(group_names):
        d = tornado_data[name]
        lo, hi = d[f"{metric}_min"], d[f"{metric}_max"]
        color = TORNADO_COLORS.get(name, "lightgray")
        ax.barh(i, hi - lo, left=lo, color=color, alpha=0.9,
                edgecolor="white", height=0.6)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(group_names)
    ax.set_xlabel(f"Change in best {label} (%)")


# Side-by-side panel, same layout as the sensitivity bar charts in section 6
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
_draw_tornado(axes[0], "rov", "E[ROV]")
_draw_tornado(axes[1], "cvar", "CVaR")
plt.tight_layout()
plt.show()


In [ ]:
# Rank groups by total swing and auto-select top 4 for deep dives
ranked_desc = sorted(tornado_data.items(), key=lambda x: -x[1]["total_swing"])
DEEP_DIVE_GROUPS = [name for name, _ in ranked_desc[:4]]

print("Parameter groups ranked by total sensitivity swing:\n")
for i, (name, data) in enumerate(ranked_desc):
    marker = " <-- deep dive" if name in DEEP_DIVE_GROUPS else ""
    print(f"  {i+1}. {name:15s}: "
          f"ROV [{data['rov_min']:+6.1f}%, {data['rov_max']:+6.1f}%]  "
          f"CVaR [{data['cvar_min']:+6.1f}%, {data['cvar_max']:+6.1f}%]"
          f"  swing={data['total_swing']:.1f}%{marker}")

## 5. CVaR Alpha — Metric Redefinition

CVaR alpha changes what CVaR *measures* (1% vs 5% vs 20% tail), not the portfolio's actual risk. Increasing alpha includes more scenarios in the tail average, pulling CVaR toward the mean. E[ROV] should be unaffected since the optimizer's first objective does not depend on alpha.

In [ ]:
group_info = PARAM_GROUPS["CVaR alpha"]
cvar_configs = group_info["configs"]
cvar_values = group_info["values"]
cvar_labels = [f"{v}%" for v in cvar_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# E[ROV] box plots
rov_data = [summary_stats[c]["best_rov"] for c in cvar_configs]
bp0 = axes[0].boxplot(rov_data, labels=cvar_labels, patch_artist=True)
for i, patch in enumerate(bp0["boxes"]):
    alpha_val = 0.9 if i == group_info["baseline_idx"] else 0.6
    patch.set_facecolor("orchid")
    patch.set_alpha(alpha_val)
axes[0].set_xlabel(group_info["xlabel"])
axes[0].set_ylabel("Best E[ROV] (MNOK)")
axes[0].set_title("E[ROV] across CVaR alpha levels")
axes[0].grid(True, alpha=0.3)

# CVaR box plots
cvar_data = [summary_stats[c]["best_cvar"] for c in cvar_configs]
bp1 = axes[1].boxplot(cvar_data, labels=cvar_labels, patch_artist=True)
for i, patch in enumerate(bp1["boxes"]):
    alpha_val = 0.9 if i == group_info["baseline_idx"] else 0.6
    patch.set_facecolor("orchid")
    patch.set_alpha(alpha_val)
axes[1].set_xlabel(group_info["xlabel"])
axes[1].set_ylabel("Best CVaR (MNOK)")
axes[1].set_title("CVaR across alpha levels (metric redefinition)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confirm E[ROV] is independent of CVaR alpha, CVaR changes mechanically
rov_means = [summary_stats[c]["best_rov"].mean() for c in cvar_configs]
cvar_means = [summary_stats[c]["best_cvar"].mean() for c in cvar_configs]
rov_spread = (max(rov_means) - min(rov_means)) / np.mean(rov_means) * 100

print(f"E[ROV] varies by {rov_spread:.1f}% across alpha levels (should be ~0%)")
print(f"CVaR increases from {min(cvar_means):,.0f} to {max(cvar_means):,.0f} MNOK "
      f"as alpha widens from {min(cvar_values)}% to {max(cvar_values)}%")

## 6. Objective Sensitivity — Box Plots per Parameter Group

For each parameter group (excluding CVaR alpha): box plots of best E[ROV] and best CVaR at each parameter value. The baseline value is highlighted with a darker fill.

In [ ]:
# Bar-chart sensitivity figures for the three parameters most central to tieback economics:
# volatility (deferral option value), host capacity (operational constraint),
# CAPEX (investment cost). Style: left panel Mean ROV, right panel CVaR_0.05,
# with the baseline drawn as a red dashed line.

TARGET_GROUPS = [
    ("Volatility",    "Price Volatility Scale"),
    ("Host capacity", "Host Capacity Scale"),
    ("CAPEX",         "CAPEX Scale"),
]


def _fmt_scale_labels(values):
    return [f"{int(round(v * 100))}%" for v in values]


def plot_sensitivity_bars(group_name, x_label_pretty, annotate_indices=None):
    info = PARAM_GROUPS[group_name]
    cfgs = info["configs"]
    vals = info["values"]
    bl_idx = info["baseline_idx"]
    labels = _fmt_scale_labels(vals)

    rov_means = np.array([summary_stats[c]["best_rov"].mean() for c in cfgs])
    cvar_means = np.array([summary_stats[c]["best_cvar"].mean() for c in cfgs])
    bl_rov = rov_means[bl_idx]
    bl_cvar = cvar_means[bl_idx]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(cfgs))

    axes[0].bar(x, rov_means, color="steelblue", alpha=0.85, edgecolor="white", width=0.7)
    axes[0].axhline(bl_rov, color="red", linestyle="--", linewidth=1.5,
                    label=f"Baseline: {bl_rov:,.0f}")
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
    axes[0].set_xlabel(x_label_pretty)
    axes[0].set_ylabel("Mean ROV (MNOK)")
    axes[0].set_title(f"Mean ROV vs {x_label_pretty}", fontsize=12, fontweight="bold")
    axes[0].legend(loc="upper left")

    axes[1].bar(x, cvar_means, color="lightsalmon", alpha=0.95, edgecolor="white", width=0.7)
    axes[1].axhline(bl_cvar, color="red", linestyle="--", linewidth=1.5,
                    label=f"Baseline: {bl_cvar:,.0f}")
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
    axes[1].set_xlabel(x_label_pretty)
    axes[1].set_ylabel(r"CVaR$_{0.05}$ (MNOK)")
    axes[1].set_title(rf"CVaR$_{{0.05}}$ vs {x_label_pretty}", fontsize=12, fontweight="bold")
    axes[1].legend(loc="upper left")

    if annotate_indices:
        for i in annotate_indices:
            axes[1].annotate(f"{cvar_means[i]:,.0f} MNOK",
                             xy=(x[i], cvar_means[i]),
                             xytext=(0, 6), textcoords="offset points",
                             ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()

    # Print the plotted values
    print(f"\n{group_name}:")
    for lab, r, c in zip(labels, rov_means, cvar_means):
        marker = " (baseline)" if labels.index(lab) == bl_idx else ""
        print(f"  {lab:5s}: ROV={r:>10,.0f} MNOK, CVaR={c:>9,.0f} MNOK{marker}")


for group_name, x_label_pretty in TARGET_GROUPS:
    # CAPEX figure: annotate the 3x and 4x bars (indices 6 and 7)
    # since their tiny values (15 MNOK, 0 MNOK) are visually indistinguishable from zero.
    annotate = [6, 7] if group_name == "CAPEX" else None
    plot_sensitivity_bars(group_name, x_label_pretty, annotate_indices=annotate)


## 7. Deep Dive — Overlaid Pareto Fronts

For the most impactful parameter groups, overlay Pareto fronts from the median-HVI run per config. This shows how the front shape and location shift with the parameter, not just the best-ROV and best-CVaR endpoints.

In [ ]:
# Load Pareto fronts for deep-dive groups + baseline
# Collect all unique config names needed
deep_dive_configs = set()
for group_name in DEEP_DIVE_GROUPS:
    deep_dive_configs.update(PARAM_GROUPS[group_name]["configs"])
deep_dive_configs.add("baseline")

print(f"Loading pickles for {len(deep_dive_configs)} configs...")

fronts_by_config = {}
for cfg_name in sorted(deep_dive_configs):
    fronts = []
    for run in configs[cfg_name]:
        result = load_run_result(run["pickle_path"])
        pf = result["pareto_front"]
        if pf:
            obj = np.array([e.get("objectives_final", e["objectives"]) for e in pf])
        else:
            obj = np.empty((0, 2))
        fronts.append(obj)
    fronts_by_config[cfg_name] = fronts

print(f"Loaded Pareto fronts for {len(fronts_by_config)} configs")

In [ ]:
# Select median-HVI run per config (for representative Pareto front)
# Need a common reference point for HVI
all_loaded_objs = [f for fronts in fronts_by_config.values() for f in fronts if len(f) > 0]
all_combined = np.vstack(all_loaded_objs)
ref_point = np.array([
    all_combined[:, 0].min() - abs(all_combined[:, 0].min()) * 0.1,
    all_combined[:, 1].min() - abs(all_combined[:, 1].min()) * 0.1,
])

median_fronts = {}
for cfg_name, fronts in fronts_by_config.items():
    hvi_vals = np.array([hypervolume_2d(f, ref_point) for f in fronts])
    median_idx = np.argmin(np.abs(hvi_vals - np.median(hvi_vals)))
    median_fronts[cfg_name] = fronts[median_idx]

In [ ]:
# Overlaid Pareto fronts for each deep-dive group
for group_name in DEEP_DIVE_GROUPS:
    group_info = PARAM_GROUPS[group_name]
    cfgs = group_info["configs"]
    vals = group_info["values"]
    bl_idx = group_info["baseline_idx"]

    fig, ax = plt.subplots(figsize=(10, 7))

    # Use a colormap for the parameter sweep
    cmap = plt.cm.viridis
    n_cfgs = len(cfgs)

    for i, (cfg, val) in enumerate(zip(cfgs, vals)):
        front = median_fronts[cfg]
        if len(front) == 0:
            continue
        # Sort by CVaR for consistent line
        sort_idx = np.argsort(front[:, 1])
        sorted_front = front[sort_idx]

        if cfg == "baseline":
            ax.plot(sorted_front[:, 1], sorted_front[:, 0], "s-",
                    color="black", markersize=5, linewidth=1.5,
                    label=f"Baseline ({val})", zorder=10)
        else:
            color = cmap(i / max(1, n_cfgs - 1))
            ax.plot(sorted_front[:, 1], sorted_front[:, 0], "o-",
                    color=color, markersize=4, linewidth=1.2,
                    label=f"{val}", alpha=0.8)

    ax.set_xlabel("CVaR (MNOK)")
    ax.set_ylabel("E[ROV] (MNOK)")
    ax.set_title(f"Pareto fronts — {group_name} sensitivity")
    ax.legend(title=group_info["xlabel"], fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Portfolio Composition Analysis

How does changing input assumptions affect *which* fields are selected, how they are assigned to hosts, when they start, and how many wells are drilled? For each config, the best-ROV solution from the median-HVI run is decoded and compared against baseline.

**Note:** Configs that change data generation parameters (field size, host capacity, CAPEX) generate different problem instances — the "same field" has different properties. Field indices and host indices are consistent, so structural comparisons (assignment, selection) remain valid.

In [ ]:
# Load pickle for remaining configs not yet loaded, and decode best-ROV allocations
# First, get problem dimensions from any result
sample_result = load_run_result(configs["baseline"][0]["pickle_path"])
n_fields = sample_result["config"]["n_fields"]
n_hosts = sample_result["config"]["n_hosts"]
max_start_year = sample_result["config"]["max_start_year"]
max_wells = sample_result["config"]["data_generation"]["max_wells"]
encoding = sample_result["config"]["pso"].get("encoding", None)
field_names = [f"F{i+1}" for i in range(n_fields)]
host_names = [f"H{i+1}" for i in range(n_hosts)]

print(f"Problem: {n_fields} fields, {n_hosts} hosts, max_start_year={max_start_year}, max_wells={max_wells}")

# For each config: find median-HVI run, decode best-ROV Pareto solution
best_rov_allocs = {}
all_config_names = sorted(configs.keys())

for cfg_name in all_config_names:
    # Load fronts if not already loaded
    if cfg_name not in fronts_by_config:
        fronts = []
        for run in configs[cfg_name]:
            result = load_run_result(run["pickle_path"])
            pf = result["pareto_front"]
            if pf:
                obj = np.array([e.get("objectives_final", e["objectives"]) for e in pf])
            else:
                obj = np.empty((0, 2))
            fronts.append(obj)
        fronts_by_config[cfg_name] = fronts

    # Find median-HVI run
    fronts = fronts_by_config[cfg_name]
    hvi_vals = np.array([hypervolume_2d(f, ref_point) for f in fronts])
    median_idx = np.argmin(np.abs(hvi_vals - np.median(hvi_vals)))

    # Load the median run's full result to get positions
    median_run = configs[cfg_name][median_idx]
    result = load_run_result(median_run["pickle_path"])
    pf = result["pareto_front"]

    if pf:
        # Find best-ROV solution (max first objective)
        objs = np.array([e.get("objectives_final", e["objectives"]) for e in pf])
        best_idx = np.argmax(objs[:, 0])
        position = pf[best_idx]["position"]
        alloc = decode_particle(position, n_fields, n_hosts, max_start_year, max_wells,
                                encoding=encoding)
        best_rov_allocs[cfg_name] = alloc

print(f"\nDecoded best-ROV allocations for {len(best_rov_allocs)} configs")

In [ ]:
def developed_set(alloc):
    """Return the set of developed field indices."""
    return frozenset(f for f in range(alloc.n_fields) if alloc.assignment[f] >= 0)


def assignment_signature(alloc):
    """Return a hashable tuple of (field_idx, host_idx) for developed fields."""
    return tuple(sorted(
        (f, alloc.assignment[f]) for f in range(alloc.n_fields)
        if alloc.assignment[f] >= 0
    ))


# Compute Jaccard similarity of field selections vs baseline
baseline_set = developed_set(best_rov_allocs["baseline"])
jaccard = {}
for cfg_name, alloc in best_rov_allocs.items():
    s = developed_set(alloc)
    union = baseline_set | s
    jaccard[cfg_name] = len(baseline_set & s) / len(union) if len(union) > 0 else 1.0

In [ ]:
# Jaccard similarity grouped bar chart
fig, ax = plt.subplots(figsize=(14, 5))

# Build bars grouped by parameter group
bar_labels = []
bar_values = []
bar_colors = []
group_boundaries = []

for group_name, group_info in PARAM_GROUPS.items():
    group_boundaries.append(len(bar_labels))
    for cfg in group_info["configs"]:
        if cfg == "baseline":
            continue
        bar_labels.append(cfg)
        bar_values.append(jaccard.get(cfg, 0.0))
        bar_colors.append(GROUP_COLORS[group_name])

x = np.arange(len(bar_labels))
bars = ax.bar(x, bar_values, color=bar_colors, alpha=0.7, edgecolor="white")
ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(bar_labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Jaccard similarity vs baseline")
ax.set_title("Field selection stability across parameter variations")
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis="y")

# Add group labels
for i, (group_name, _) in enumerate(PARAM_GROUPS.items()):
    start = group_boundaries[i]
    n_bars = len([c for c in PARAM_GROUPS[group_name]["configs"] if c != "baseline"])
    end = start + n_bars - 1
    mid = (start + end) / 2
    ax.text(mid, 1.05, group_name, ha="center", fontsize=8, fontweight="bold",
            color=GROUP_COLORS[group_name])

plt.tight_layout()
plt.show()

In [ ]:
# Host assignment heatmap: rows = configs (grouped), cols = fields
# Values: host index (0, 1, 2) or -1 (not developed)
# Build ordered config list grouped by parameter group
ordered_configs = []
ordered_labels = []
ordered_group_colors = []
for group_name, group_info in PARAM_GROUPS.items():
    for cfg in group_info["configs"]:
        if cfg == "baseline" and cfg in [c for c in ordered_configs]:
            continue
        if cfg not in best_rov_allocs:
            continue
        ordered_configs.append(cfg)
        ordered_labels.append(cfg)
        ordered_group_colors.append(GROUP_COLORS[group_name])

# Build assignment matrix
assign_matrix = np.full((len(ordered_configs), n_fields), np.nan)
for i, cfg in enumerate(ordered_configs):
    alloc = best_rov_allocs[cfg]
    for f in range(n_fields):
        if alloc.assignment[f] >= 0:
            assign_matrix[i, f] = alloc.assignment[f]

fig, ax = plt.subplots(figsize=(12, 8))

# Custom colormap: white for NaN, distinct colors for hosts
from matplotlib.colors import ListedColormap, BoundaryNorm
host_colors = ["steelblue", "seagreen", "coral"]
cmap_hosts = ListedColormap(host_colors[:n_hosts])
bounds = np.arange(-0.5, n_hosts + 0.5, 1)
norm = BoundaryNorm(bounds, cmap_hosts.N)

im = ax.imshow(assign_matrix, aspect="auto", cmap=cmap_hosts, norm=norm, interpolation="nearest")
ax.set_xticks(range(n_fields))
ax.set_xticklabels(field_names)
ax.set_yticks(range(len(ordered_configs)))
ax.set_yticklabels(ordered_labels, fontsize=8)
ax.set_xlabel("Field")
ax.set_ylabel("Configuration")
ax.set_title("Host assignment across parameter variations (best-ROV solution)")

# Add host legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=host_colors[h], label=host_names[h]) for h in range(n_hosts)]
legend_elements.append(Patch(facecolor="white", edgecolor="gray", label="Not developed"))
ax.legend(handles=legend_elements, loc="upper right", fontsize=9)

# Add host index text in each cell
for i in range(len(ordered_configs)):
    for f in range(n_fields):
        val = assign_matrix[i, f]
        if not np.isnan(val):
            ax.text(f, i, host_names[int(val)], ha="center", va="center", fontsize=7,
                    color="white", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Start year and well count heatmaps
timing_matrix = np.full((len(ordered_configs), n_fields), np.nan)
wells_matrix = np.full((len(ordered_configs), n_fields), np.nan)
for i, cfg in enumerate(ordered_configs):
    alloc = best_rov_allocs[cfg]
    for f in range(n_fields):
        if alloc.assignment[f] >= 0:
            timing_matrix[i, f] = alloc.start_year[f]
            wells_matrix[i, f] = alloc.n_wells[f]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Start year
im0 = axes[0].imshow(timing_matrix, aspect="auto", cmap="YlOrRd", interpolation="nearest")
axes[0].set_xticks(range(n_fields))
axes[0].set_xticklabels(field_names)
axes[0].set_yticks(range(len(ordered_configs)))
axes[0].set_yticklabels(ordered_labels, fontsize=8)
axes[0].set_xlabel("Field")
axes[0].set_title("Start year")
plt.colorbar(im0, ax=axes[0], label="Year", shrink=0.8)
# Add text values
for i in range(len(ordered_configs)):
    for f in range(n_fields):
        val = timing_matrix[i, f]
        if not np.isnan(val):
            axes[0].text(f, i, f"{int(val)}", ha="center", va="center", fontsize=6)

# Well count
im1 = axes[1].imshow(wells_matrix, aspect="auto", cmap="YlGnBu", interpolation="nearest")
axes[1].set_xticks(range(n_fields))
axes[1].set_xticklabels(field_names)
axes[1].set_yticks(range(len(ordered_configs)))
axes[1].set_yticklabels(ordered_labels, fontsize=8)
axes[1].set_xlabel("Field")
axes[1].set_title("Well count")
plt.colorbar(im1, ax=axes[1], label="Wells", shrink=0.8)
# Add text values
for i in range(len(ordered_configs)):
    for f in range(n_fields):
        val = wells_matrix[i, f]
        if not np.isnan(val):
            axes[1].text(f, i, f"{int(val)}", ha="center", va="center", fontsize=6)

plt.tight_layout()
plt.show()

## 9. Deep Dive — Composition Changes for Top Groups

For the most impactful parameter groups: field selection frequency across the full Pareto front at each parameter value, and how timing/well count shift with the parameter.

In [ ]:
# Field selection frequency across full Pareto fronts for deep-dive groups
# For each config, decode ALL Pareto solutions from the median-HVI run,
# then compute per-field selection frequency.
selection_freq_by_config = {}

for group_name in DEEP_DIVE_GROUPS:
    group_info = PARAM_GROUPS[group_name]
    for cfg in group_info["configs"]:
        if cfg in selection_freq_by_config:
            continue

        # Find median-HVI run
        fronts = fronts_by_config[cfg]
        hvi_vals = np.array([hypervolume_2d(f, ref_point) for f in fronts])
        median_idx = np.argmin(np.abs(hvi_vals - np.median(hvi_vals)))

        # Load and decode all Pareto solutions
        median_run = configs[cfg][median_idx]
        result = load_run_result(median_run["pickle_path"])
        pf = result["pareto_front"]

        if pf:
            freq = np.zeros(n_fields)
            for entry in pf:
                alloc = decode_particle(entry["position"], n_fields, n_hosts,
                                        max_start_year, max_wells, encoding=encoding)
                for f in range(n_fields):
                    if alloc.assignment[f] >= 0:
                        freq[f] += 1
            freq /= len(pf)
        else:
            freq = np.zeros(n_fields)

        selection_freq_by_config[cfg] = freq

print(f"Computed selection frequencies for {len(selection_freq_by_config)} configs")

In [ ]:
# Field selection frequency plots for top 2 deep-dive groups
for group_name in DEEP_DIVE_GROUPS[:2]:
    group_info = PARAM_GROUPS[group_name]
    cfgs = group_info["configs"]
    vals = group_info["values"]
    n_cfgs = len(cfgs)

    fig, ax = plt.subplots(figsize=(12, 5))

    bar_width = 0.8 / n_cfgs
    x = np.arange(n_fields)

    cmap = plt.cm.viridis
    for i, (cfg, val) in enumerate(zip(cfgs, vals)):
        freq = selection_freq_by_config[cfg]
        offset = (i - n_cfgs / 2 + 0.5) * bar_width
        color = "black" if cfg == "baseline" else cmap(i / max(1, n_cfgs - 1))
        ax.bar(x + offset, freq, bar_width, label=f"{val}",
               color=color, alpha=0.8, edgecolor="white")

    ax.set_xticks(x)
    ax.set_xticklabels(field_names)
    ax.set_xlabel("Field")
    ax.set_ylabel("Selection frequency in Pareto front")
    ax.set_title(f"Field selection frequency — {group_name} sensitivity")
    ax.set_ylim(0, 1.15)
    ax.axhline(0.8, color="gray", linewidth=0.8, linestyle=":", alpha=0.5, label="Core threshold")
    ax.legend(title=group_info["xlabel"], fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

In [ ]:
# Timing and well count trends across parameter values (exploratory, not saved)
for group_name in DEEP_DIVE_GROUPS:
    group_info = PARAM_GROUPS[group_name]
    cfgs = group_info["configs"]
    vals = group_info["values"]

    # Build arrays: (n_configs, n_fields) for start_year and n_wells
    sy_arr = np.full((len(cfgs), n_fields), np.nan)
    nw_arr = np.full((len(cfgs), n_fields), np.nan)
    for i, cfg in enumerate(cfgs):
        alloc = best_rov_allocs[cfg]
        for f in range(n_fields):
            if alloc.assignment[f] >= 0:
                sy_arr[i, f] = alloc.start_year[f]
                nw_arr[i, f] = alloc.n_wells[f]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for f in range(n_fields):
        axes[0].plot(vals, sy_arr[:, f], "o-", markersize=4, label=field_names[f], alpha=0.7)
        axes[1].plot(vals, nw_arr[:, f], "o-", markersize=4, label=field_names[f], alpha=0.7)

    axes[0].set_xlabel(group_info["xlabel"])
    axes[0].set_ylabel("Start year")
    axes[0].set_title(f"{group_name} — Start year per field")
    axes[0].legend(fontsize=8, ncol=3)
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel(group_info["xlabel"])
    axes[1].set_ylabel("Well count")
    axes[1].set_title(f"{group_name} — Well count per field")
    axes[1].legend(fontsize=8, ncol=3)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 10. Robustness Summary

Which findings from E5.1 hold across parameter ranges? Which are sensitive?

In [ ]:
# Core field stability: which fields remain core (>80% selection) across all configs?
print("=== Core Field Stability ===\n")

for group_name in PARAM_GROUPS:
    group_info = PARAM_GROUPS[group_name]
    cfgs = group_info["configs"]

    # Get selection frequency for each config (use best-ROV alloc as proxy if
    # full selection frequency not computed for this group)
    core_sets = []
    for cfg in cfgs:
        if cfg in selection_freq_by_config:
            freq = selection_freq_by_config[cfg]
            core = frozenset(f for f in range(n_fields) if freq[f] >= 0.8)
        elif cfg in best_rov_allocs:
            # All developed fields are "core" in the best-ROV solution
            core = developed_set(best_rov_allocs[cfg])
        else:
            continue
        core_sets.append(core)

    if core_sets:
        always_core = set.intersection(*[set(s) for s in core_sets])
        ever_core = set.union(*[set(s) for s in core_sets])
        changing = ever_core - always_core
        print(f"  {group_name:15s}: "
              f"always core = {{{', '.join(field_names[f] for f in sorted(always_core))}}}  "
              f"changing = {{{', '.join(field_names[f] for f in sorted(changing)) or 'none'}}}")

In [ ]:
# Assignment topology stability: how many distinct host assignments appear?
print("=== Assignment Topology Stability ===\n")

baseline_sig = assignment_signature(best_rov_allocs["baseline"])
topology_counts = defaultdict(list)
for cfg_name, alloc in best_rov_allocs.items():
    sig = assignment_signature(alloc)
    topology_counts[sig].append(cfg_name)

print(f"Total distinct topologies: {len(topology_counts)} across {len(best_rov_allocs)} configs\n")

for sig, cfg_list in sorted(topology_counts.items(), key=lambda x: -len(x[1])):
    is_baseline = " (BASELINE)" if sig == baseline_sig else ""
    print(f"  Topology used by {len(cfg_list)} configs{is_baseline}:")
    for cfg in cfg_list:
        print(f"    - {cfg}")
    # Show the assignment
    first_alloc = best_rov_allocs[cfg_list[0]]
    assign_str = ", ".join(
        f"{field_names[f]}→{host_names[first_alloc.assignment[f]]}"
        for f in range(n_fields) if first_alloc.assignment[f] >= 0
    )
    print(f"    Assignment: {assign_str}\n")

In [ ]:
# Summary table: per group robustness
print("=== Robustness Summary by Parameter Group ===\n")
print(f"{'Group':<15s}  {'Jaccard range':<18s}  {'Topology change':<18s}  {'Fields developed':>17s}")
print("-" * 75)

for group_name, group_info in PARAM_GROUPS.items():
    cfgs = [c for c in group_info["configs"] if c != "baseline" and c in jaccard]
    if not cfgs:
        continue
    j_vals = [jaccard[c] for c in cfgs]
    j_range = f"{min(j_vals):.2f} — {max(j_vals):.2f}"

    # Check topology
    topo_changes = sum(
        1 for c in cfgs
        if c in best_rov_allocs and assignment_signature(best_rov_allocs[c]) != baseline_sig
    )
    topo_str = f"{topo_changes}/{len(cfgs)} differ" if topo_changes > 0 else "all same"

    # Fields developed
    n_dev = [len(developed_set(best_rov_allocs[c])) for c in cfgs if c in best_rov_allocs]
    dev_str = f"{min(n_dev)}–{max(n_dev)}" if min(n_dev) != max(n_dev) else str(min(n_dev))

    print(f"{group_name:<15s}  {j_range:<18s}  {topo_str:<18s}  {dev_str:>17s}")

### Interpretation

**Robust conclusions from E5.1:**
- *(To be filled after running the notebook)*

**Sensitive conclusions:**
- *(To be filled after running the notebook)*

**Most critical input assumptions:**
- *(To be filled after running the notebook)*

## 11. CAPEX Component Analysis

CAPEX has three components with different cost drivers:
- **Drilling**: `(well_cost_base + depth_factor × depth) × n_wells` — scales with well count and reservoir depth
- **Subsea**: `subsea_base` — flat cost per tieback, independent of distance or depth
- **Pipeline**: `pipeline_cost_per_km² × distance²` — quadratic with field-to-host distance

This section shows (1) the baseline CAPEX breakdown per field, (2) how the mix shifts across CAPEX-related configs, and (3) which individual component has the most influence on portfolio value.

In [ ]:
# Per-field CAPEX breakdown at baseline
# Regenerate baseline data to get CostData (well_cost_per_well, subsea_base, pipeline_cost)
baseline_config = load_config("../config/default.yaml")
field_data, host_data, cost_data = generate_data(baseline_config)

# Get the baseline best-ROV allocation
bl_alloc = best_rov_allocs["baseline"]

# Compute per-field CAPEX components for the baseline allocation
drilling_per_field = []
subsea_per_field = []
pipeline_per_field = []
developed_fields = []

for f in range(n_fields):
    h = bl_alloc.assignment[f]
    if h < 0:
        continue
    developed_fields.append(f)
    nw = bl_alloc.n_wells[f]
    drilling_per_field.append(cost_data.well_cost_per_well[f] * nw)
    subsea_per_field.append(cost_data.subsea_base)
    pipeline_per_field.append(cost_data.pipeline_cost[f, h])

drilling_arr = np.array(drilling_per_field)
subsea_arr = np.array(subsea_per_field)
pipeline_arr = np.array(pipeline_per_field)
total_arr = drilling_arr + subsea_arr + pipeline_arr
dev_labels = [field_names[f] for f in developed_fields]

# Stacked horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(developed_fields))

ax.barh(y, drilling_arr, color="darkred", alpha=0.8, label="Drilling")
ax.barh(y, subsea_arr, left=drilling_arr, color="teal", alpha=0.8, label="Subsea")
ax.barh(y, pipeline_arr, left=drilling_arr + subsea_arr, color="mediumpurple", alpha=0.8,
        label="Pipeline")

# Add total labels
for i, total in enumerate(total_arr):
    ax.text(total + 50, i, f"{total:,.0f}", va="center", fontsize=8)

ax.set_yticks(y)
ax.set_yticklabels(dev_labels)
ax.set_xlabel("CAPEX (MNOK)")
ax.set_title("CAPEX breakdown per field (baseline, best-ROV allocation)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

# Print summary
print(f"\nPortfolio totals: Drilling={drilling_arr.sum():,.0f}, "
      f"Subsea={subsea_arr.sum():,.0f}, Pipeline={pipeline_arr.sum():,.0f}, "
      f"Total={total_arr.sum():,.0f} MNOK")
print(f"Shares: Drilling={drilling_arr.sum()/total_arr.sum()*100:.1f}%, "
      f"Subsea={subsea_arr.sum()/total_arr.sum()*100:.1f}%, "
      f"Pipeline={pipeline_arr.sum()/total_arr.sum()*100:.1f}%")

In [ ]:
# Portfolio-level CAPEX composition across CAPEX-related configs
# For each config, compute total portfolio CAPEX using that config's cost parameters
# and the best-ROV allocation from that config's optimization run.
from src.experiments.configs import get_experiment_configs

capex_groups = ["CAPEX", "Well cost", "Subsea", "Pipeline"]
capex_config_names = set()
for g in capex_groups:
    capex_config_names.update(c for c in PARAM_GROUPS[g]["configs"] if c != "baseline")
capex_config_names.add("baseline")

# Get the overrides for each config
all_e5_2_configs = dict(get_experiment_configs("E5.2"))

# Compute CAPEX decomposition for each config
capex_decomp = {}
for cfg_name in sorted(capex_config_names):
    overrides = all_e5_2_configs.get(cfg_name, {})
    cfg = load_config("../config/default.yaml", overrides=overrides)
    _, _, cd = generate_data(cfg)

    # Use the best-ROV allocation for this config (if available)
    alloc = best_rov_allocs.get(cfg_name)
    if alloc is None:
        continue

    drill_total, subsea_total, pipe_total = 0.0, 0.0, 0.0
    for f in range(n_fields):
        h = alloc.assignment[f]
        if h < 0:
            continue
        drill_total += cd.well_cost_per_well[f] * alloc.n_wells[f]
        subsea_total += cd.subsea_base
        pipe_total += cd.pipeline_cost[f, h]

    capex_decomp[cfg_name] = {
        "drilling": drill_total,
        "subsea": subsea_total,
        "pipeline": pipe_total,
        "total": drill_total + subsea_total + pipe_total,
    }

# Stacked bar chart: one bar per config, ordered by group then by value
fig, ax = plt.subplots(figsize=(16, 6))

bar_configs = []
for g in capex_groups:
    for cfg in PARAM_GROUPS[g]["configs"]:
        if cfg not in capex_decomp:
            continue
        if cfg in [c for c in bar_configs]:
            continue
        bar_configs.append(cfg)

x = np.arange(len(bar_configs))
drill_vals = [capex_decomp[c]["drilling"] for c in bar_configs]
subsea_vals = [capex_decomp[c]["subsea"] for c in bar_configs]
pipe_vals = [capex_decomp[c]["pipeline"] for c in bar_configs]

ax.bar(x, drill_vals, color="darkred", alpha=0.8, label="Drilling")
ax.bar(x, subsea_vals, bottom=drill_vals, color="teal", alpha=0.8, label="Subsea")
ax.bar(x, pipe_vals, bottom=[d + s for d, s in zip(drill_vals, subsea_vals)],
       color="mediumpurple", alpha=0.8, label="Pipeline")

ax.set_xticks(x)
ax.set_xticklabels(bar_configs, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Total portfolio CAPEX (MNOK)")
ax.set_title("CAPEX composition across cost parameter variations")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Per-component sensitivity comparison
# Compare E[ROV] and CVaR % change from scaling each CAPEX component independently
# Only configs that have results are included
component_groups = ["Well cost", "Subsea", "Pipeline"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for group_name in component_groups:
    group_info = PARAM_GROUPS[group_name]
    cfgs = group_info["configs"]
    vals = group_info["values"]
    color = GROUP_COLORS[group_name]

    # Collect % changes for configs that have results
    plot_vals = []
    plot_rov = []
    plot_cvar = []
    for cfg, val in zip(cfgs, vals):
        if cfg == "baseline":
            plot_vals.append(val)
            plot_rov.append(0.0)
            plot_cvar.append(0.0)
        elif cfg in summary_stats:
            plot_vals.append(val)
            plot_rov.append(pct_change(summary_stats[cfg]["best_rov"].mean(), bl_rov))
            plot_cvar.append(pct_change(summary_stats[cfg]["best_cvar"].mean(), bl_cvar))

    if plot_vals:
        axes[0].plot(plot_vals, plot_rov, "o-", color=color, markersize=6,
                     linewidth=2, label=group_name, alpha=0.8)
        axes[1].plot(plot_vals, plot_cvar, "o-", color=color, markersize=6,
                     linewidth=2, label=group_name, alpha=0.8)

# Add combined CAPEX for comparison
capex_info = PARAM_GROUPS["CAPEX"]
capex_vals_plot = []
capex_rov_plot = []
capex_cvar_plot = []
for cfg, val in zip(capex_info["configs"], capex_info["values"]):
    if cfg == "baseline":
        capex_vals_plot.append(val)
        capex_rov_plot.append(0.0)
        capex_cvar_plot.append(0.0)
    elif cfg in summary_stats:
        capex_vals_plot.append(val)
        capex_rov_plot.append(pct_change(summary_stats[cfg]["best_rov"].mean(), bl_rov))
        capex_cvar_plot.append(pct_change(summary_stats[cfg]["best_cvar"].mean(), bl_cvar))

if capex_vals_plot:
    axes[0].plot(capex_vals_plot, capex_rov_plot, "s--", color="slategray", markersize=6,
                 linewidth=2, label="All CAPEX", alpha=0.8)
    axes[1].plot(capex_vals_plot, capex_cvar_plot, "s--", color="slategray", markersize=6,
                 linewidth=2, label="All CAPEX", alpha=0.8)

for ax, label in [(axes[0], "E[ROV]"), (axes[1], "CVaR")]:
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_xlabel("Cost scale factor")
    ax.set_ylabel(f"Change in best {label} (%)")
    ax.set_title(f"{label} — per-component CAPEX sensitivity")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()